In [1]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from collections import Counter
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,recall_score,precision_score,f1_score
import joblib

In [19]:
df = pd.read_csv("../data/Processed/churn_cleaned.csv")
df.head()

,Age,Gender,Tenure,Usage Frequency,Total Spend,Churn,avg_spend,engagement_score,support_pressure,payment_risk,inactive_risk,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly,tenure_group_1-2yr,tenure_group_2-4yr,tenure_group_4+yr
0,30.0,0,39.0,14.0,932.0,1.0,23.300000,546.0,0.125000,90.0,17.0,0,1,0,0,False,True,False
1,65.0,0,49.0,1.0,557.0,1.0,11.140000,49.0,0.200000,80.0,6.0,0,0,1,0,False,False,True
2,55.0,0,14.0,4.0,185.0,1.0,12.333333,56.0,0.400000,108.0,3.0,0,0,0,1,True,False,False
3,58.0,1,38.0,21.0,661.0,1.0,16.948718,798.0,0.179487,49.0,29.0,0,1,1,0,False,True,False
4,23.0,1,32.0,20.0,617.0,1.0,18.696970,640.0,0.151515,40.0,20.0,0,0,1,0,False,True,False


In [20]:

df = pd.read_csv("../data/Processed/churn_cleaned.csv")

In [21]:
leakage_cols = [
    'inactive_risk',
    'payment_risk',
    'high_value',
    'engagement_score',
    'support_pressure',
    'avg_spend'
]

df = df.drop(columns=leakage_cols, errors='ignore')

# Remove ID column
df = df.drop(columns=['CustomerID'], errors='ignore')

In [22]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps class balance
)

In [24]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns (VERY IMPORTANT)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [25]:
scaler = StandardScaler()

feature_names = X_train.columns

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
X_train = pd.DataFrame(X_train,columns=feature_names)
X_test = pd.DataFrame(X_test,columns=feature_names)

In [27]:

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
rf = RandomForestClassifier(n_estimators=100,
                            max_depth=5, 
                            class_weight='balanced',
                            random_state=0)

In [28]:
counter = Counter(y_train)
scale_pos_weight = counter[0]/counter[1]

xgb = XGBClassifier(n_estimators=100, 
                    use_label_encoder=False,
                    learning_rate=0.1,
                    scale_pos_weight=scale_pos_weight,
                    eval_metric='logloss')

In [29]:
def evaluate(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\nMODEL: {model.__class__.__name__}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("F1 Score:",f1_score(y_test, y_pred))

In [30]:
evaluate(lr)
evaluate(rf)
evaluate(xgb)


MODEL: LogisticRegression
[[33328  4839]
 [13104 36896]]
              precision    recall  f1-score   support

         0.0       0.72      0.87      0.79     38167
         1.0       0.88      0.74      0.80     50000

    accuracy                           0.80     88167
   macro avg       0.80      0.81      0.80     88167
weighted avg       0.81      0.80      0.80     88167

F1 Score: 0.8044039897530931

MODEL: RandomForestClassifier
[[38164     3]
 [11110 38890]]
              precision    recall  f1-score   support

         0.0       0.77      1.00      0.87     38167
         1.0       1.00      0.78      0.87     50000

    accuracy                           0.87     88167
   macro avg       0.89      0.89      0.87     88167
weighted avg       0.90      0.87      0.87     88167

F1 Score: 0.8749845319653966


c:\Users\LENOVO\Downloads\Zidio project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:40:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



MODEL: XGBClassifier
[[38157    10]
 [10037 39963]]
              precision    recall  f1-score   support

         0.0       0.79      1.00      0.88     38167
         1.0       1.00      0.80      0.89     50000

    accuracy                           0.89     88167
   macro avg       0.90      0.90      0.89     88167
weighted avg       0.91      0.89      0.89     88167

F1 Score: 0.8883331666166516


In [31]:
# final model
churn_model = xgb
churn_model.fit(X_train,y_train)


c:\Users\LENOVO\Downloads\Zidio project\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:40:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [35]:
print(churn_model.feature_names_in_)

['Age' 'Gender' 'Tenure' 'Usage Frequency' 'Total Spend'
 'Subscription Type_Premium' 'Subscription Type_Standard'
 'Contract Length_Monthly' 'Contract Length_Quarterly'
 'tenure_group_1-2yr' 'tenure_group_2-4yr' 'tenure_group_4+yr']


In [33]:
joblib.dump(churn_model,"../models/churn_model.pkl")
joblib.dump(scaler,"../models/scaler.pkl")

['../models/scaler.pkl']